In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

def load_and_preprocess_data(filepath):
    """Loads SNF billing data and scales features for the model."""
    df = pd.read_csv(filepath)

    # Selecting clinical and billing features for anomaly detection
    features = ['Acuity_Score', 'ADL_Score', 'Comorbidities', 'Therapy_Hours_Billed']
    X = df[features]

    # Standardization is crucial for distance/density-based algorithms
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return df, X_scaled

def detect_anomalies(df, X_scaled, contamination_rate=0.15):
    """Applies Isolation Forest to identify clinical/billing misalignments."""
    # Initialize Isolation Forest
    # Random_state ensures reproducibility.
    model = IsolationForest(
        n_estimators=100,
        max_samples='auto',
        contamination=contamination_rate,
        random_state=42
    )

    # Fit the model and predict anomalies (-1 indicates an anomaly, 1 is normal)
    df['Anomaly_Label'] = model.fit_predict(X_scaled)

    # Calculate anomaly scores (lower scores indicate higher abnormality)
    df['Anomaly_Score'] = model.decision_function(X_scaled)

    # Map numerical labels to human-readable text
    df['Status'] = df['Anomaly_Label'].map({1: 'Normal', -1: 'Review Required'})

    return df

def visualize_results(df, output_filename='anomaly_visualization.png'):
    """Generates a scatter plot highlighting anomalous billing patterns."""
    plt.figure(figsize=(10, 6))

    # Create a scatter plot: Acuity vs Therapy Hours
    sns.scatterplot(
        data=df,
        x='Acuity_Score',
        y='Therapy_Hours_Billed',
        hue='Status',
        palette={'Normal': '#2ecc71', 'Review Required': '#e74c3c'},
        s=100,
        edgecolor='black',
        alpha=0.8
    )

    # Annotate the anomalies for clarity
    anomalies = df[df['Status'] == 'Review Required']
    for _, row in anomalies.iterrows():
        plt.annotate(
            f"ID:{row['Patient_ID']}",
            (row['Acuity_Score'], row['Therapy_Hours_Billed']),
            textcoords="offset points",
            xytext=(0,10),
            ha='center',
            fontsize=9,
            weight='bold'
        )

    plt.title('SNF Billing Anomaly Detection: Clinical Acuity vs. Therapy Hours', fontsize=14, weight='bold')
    plt.xlabel('Patient Acuity Score (1 = Minor, 10 = Severe)', fontsize=12)
    plt.ylabel('Therapy Hours Billed', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)

    # Save and display
    plt.tight_layout()
    plt.savefig(output_filename, dpi=300)
    print(f"Visualization saved successfully as '{output_filename}'")

if __name__ == "__main__":
    # Define file paths
    input_file = 'snf_billing_data.csv'

    # Execute Pipeline
    print("Initializing SNF Anomaly Detection Engine...")
    try:
        data, scaled_features = load_and_preprocess_data(input_file)
        processed_data = detect_anomalies(data, scaled_features, contamination_rate=0.15)

        print("\n--- Flagged Claims for Manual Review ---")
        flagged_claims = processed_data[processed_data['Status'] == 'Review Required']
        print(flagged_claims[['Patient_ID', 'Acuity_Score', 'Therapy_Hours_Billed', 'Anomaly_Score']].to_string(index=False))

        visualize_results(processed_data)

    except FileNotFoundError:
        print(f"Error: Please ensure '{input_file}' is in the same directory as this script.")

KeyboardInterrupt: 